## Why BART After LSTM?
### LSTM is like a student who learned summarization from scratch — it works but gives rough, broken output. BART is like a professor who already mastered summarization from reading millions of articles — we just show him a few more examples to adjust.

In [1]:
#!pip install transformers datasets torch rouge-score

In [2]:
import torch
from transformers import BartTokenizer, BartForConditionalGeneration
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

c:\Users\manis\anaconda3\envs\medirepo\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load BART (Pre-trained)

In [ ]:
MODEL_NAME = "facebook/bart-large-cnn"
# This version is ALREADY fine-tuned on CNN data
# So its literally built for what we need

tokenizer = BartTokenizer.from_pretrained(MODEL_NAME)
model     = BartForConditionalGeneration.from_pretrained(MODEL_NAME)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = model.to(DEVICE)

print("BART loaded successfully!")
print(f"Running on: {DEVICE}")

## What just happened:

- BartTokenizer → handles all word to number conversion automatically
- BartForConditionalGeneration → this IS the full encoder + decoder BART model
- .from_pretrained() → downloads the model weights from HuggingFace



### Load & Prepare Dataset

In [ ]:
# Load CNN dataset (same as LSTM)
dataset = load_dataset("cnn_dailymail", "3.0.0")

# We use small sample — fine-tuning doesnt need all 300k
train_data = dataset['train'].select(range(5000))
val_data   = dataset['validation'].select(range(500))

print(f"Train samples : {len(train_data)}")
print(f"Val samples   : {len(val_data)}")

Train samples : 5000
Val samples   : 500


### Tokenize the Data

In [ ]:
# BART has limits on input length
MAX_INPUT_LEN  = 512   # max tokens for article
MAX_TARGET_LEN = 128   # max tokens for summary

def tokenize_data(batch):
    # Tokenize articles (input)
    inputs = tokenizer(
        batch['article'],
        max_length    = MAX_INPUT_LEN,
        truncation    = True,       # cut if too long
        padding       = 'max_length', # pad if too short
        return_tensors= 'pt'
    )

    # Tokenize summaries (target/output)
    targets = tokenizer(
        batch['highlights'],
        max_length    = MAX_TARGET_LEN,
        truncation    = True,
        padding       = 'max_length',
        return_tensors= 'pt'
    )

    inputs['labels'] = targets['input_ids']
    return inputs

print("Sample tokenized:")
sample = tokenize_data(dataset['train'][0:1])
print(f"Input shape  : {sample['input_ids'].shape}")
print(f"Labels shape : {sample['labels'].shape}")

Sample tokenized:
Input shape  : torch.Size([1, 512])
Labels shape : torch.Size([1, 128])


### Build Dataset Class

In [ ]:
class CNNDataset(Dataset):
    def __init__(self, data, tokenizer, max_input, max_target):
        self.data       = data
        self.tokenizer  = tokenizer
        self.max_input  = max_input
        self.max_target = max_target

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        article  = self.data[idx]['article']
        summary  = self.data[idx]['highlights']

        # Tokenize input article
        inputs = self.tokenizer(
            article,
            max_length    = self.max_input,
            truncation    = True,
            padding       = 'max_length',
            return_tensors= 'pt'
        )

        # Tokenize target summary
        targets = self.tokenizer(
            summary,
            max_length    = self.max_target,
            truncation    = True,
            padding       = 'max_length',
            return_tensors= 'pt'
        )

        return {
            'input_ids'      : inputs['input_ids'].squeeze(),
            'attention_mask' : inputs['attention_mask'].squeeze(),
            # -100 tells model to ignore PAD tokens in loss
            'labels'         : targets['input_ids'].squeeze().masked_fill(
                                targets['input_ids'].squeeze() == tokenizer.pad_token_id, -100
                               )
        }

# Create datasets
train_dataset = CNNDataset(train_data, tokenizer, MAX_INPUT_LEN, MAX_TARGET_LEN)
val_dataset   = CNNDataset(val_data,   tokenizer, MAX_INPUT_LEN, MAX_TARGET_LEN)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=4, shuffle=False)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")

Train batches : 1250
Val batches   : 125


### Why batch_size=4?
BART is a huge model. Bigger batch = more RAM needed. 4 is safe for most machines.

#  Fine-Tune BART

In [ ]:
# Optimizer — AdamW works best for transformers
optimizer = AdamW(model.parameters(), lr=2e-5)
# Small learning rate because model is already pre-trained
# We just nudge it, not retrain from scratch

def train_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0

    for batch_idx, batch in enumerate(loader):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['labels'].to(DEVICE)

        # Forward pass
        # BART calculates loss internally when labels are provided
        outputs = model(
            input_ids      = input_ids,
            attention_mask = attention_mask,
            labels         = labels
        )

        loss = outputs.loss  # BART gives us loss directly

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

        if batch_idx % 100 == 0:
            print(f"  Batch {batch_idx}/{len(loader)} | Loss: {loss.item():.4f}")

    return total_loss / len(loader)


def validate(model, loader):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels         = batch['labels'].to(DEVICE)

            outputs = model(
                input_ids      = input_ids,
                attention_mask = attention_mask,
                labels         = labels
            )

            total_loss += outputs.loss.item()

    return total_loss / len(loader)


# Training loop
N_EPOCHS = 3  # 3 is enough since model is pre-trained

for epoch in range(N_EPOCHS):
    print(f"\nEpoch {epoch+1}/{N_EPOCHS}")

    train_loss = train_epoch(model, train_loader, optimizer)
    val_loss   = validate(model, val_loader)

    print(f"Train Loss : {train_loss:.4f}")
    print(f"Val Loss   : {val_loss:.4f}")

# Save fine-tuned model
model.save_pretrained("bart_summarizer")
tokenizer.save_pretrained("bart_summarizer")
print("\nModel saved to bart_summarizer/")


Epoch 1/3


: 

## Generate Summary (Test It)

In [ ]:
def generate_summary(text, model, tokenizer, max_length=150):
    model.eval()

    # Tokenize input
    inputs = tokenizer(
        text,
        max_length     = MAX_INPUT_LEN,
        truncation     = True,
        return_tensors = 'pt'
    ).to(DEVICE)

    # Generate summary
    with torch.no_grad():
        summary_ids = model.generate(
            inputs['input_ids'],
            attention_mask    = inputs['attention_mask'],
            max_length        = max_length,
            min_length        = 30,
            num_beams         = 4,    # beam search — tries 4 paths, picks best
            length_penalty    = 2.0,  # encourages longer summaries
            early_stopping    = True
        )

    # Decode numbers back to words
    summary = tokenizer.decode(
        summary_ids[0],
        skip_special_tokens = True  # removes <SOS> <EOS> etc
    )

    return summary


# Test on medical text
medical_report = """
The patient is a 56 year old male who presented to the emergency department 
with complaints of chest pain, shortness of breath, and dizziness for the 
past 3 days. Physical examination revealed elevated blood pressure of 160/100 
mmHg and irregular heartbeat. Laboratory tests confirmed elevated blood sugar 
levels. The patient was diagnosed with Type 2 Diabetes Mellitus and 
hypertension. Treatment plan includes metformin 500mg twice daily, 
lisinopril 10mg once daily, and dietary modifications. Patient was advised 
to follow up in 2 weeks.
"""

summary = generate_summary(medical_report, model, tokenizer)
print(f"\nOriginal Length : {len(medical_report.split())} words")
print(f"Summary Length  : {len(summary.split())} words")
print(f"\nGenerated Summary:\n{summary}")

## Evaluate with ROUGE Score

In [ ]:
from rouge_score import rouge_scorer

def evaluate_rouge(original_summary, generated_summary):
    scorer = rouge_scorer.RougeScorer(
        ['rouge1', 'rouge2', 'rougeL'],
        use_stemmer=True
    )
    scores = scorer.score(original_summary, generated_summary)

    print("\nROUGE Scores:")
    print(f"ROUGE-1 : {scores['rouge1'].fmeasure:.4f}")
    print(f"ROUGE-2 : {scores['rouge2'].fmeasure:.4f}")
    print(f"ROUGE-L : {scores['rougeL'].fmeasure:.4f}")
    return scores


# Compare BART summary vs actual summary
actual_summary = "56-year-old male diagnosed with Type 2 Diabetes and hypertension. Prescribed metformin and lisinopril."

evaluate_rouge(actual_summary, summary)

## 📊 ROUGE Scores — What They Mean

| Score    | What it Checks                          |
|----------|------------------------------------------|
| ROUGE-1  | How many single words match              |
| ROUGE-2  | How many 2-word phrases match            |
| ROUGE-L  | How long the matching sequences are      |

> ✅ Closer to **1.0** = Better performance

---

## ⚖️ LSTM vs BART — Output Comparison

**Input:** Same medical report

---

### 🔷 LSTM Output (Basic)
patient has diabetes and high blood pressure given medication  
*(rough, grammatically weak)*

---

### 🔷 BART Output (Refined)
A 56-year-old male was diagnosed with Type 2 Diabetes and hypertension.  
He was prescribed metformin and lisinopril with dietary modifications and follow-up in 2 weeks.  

*(clean, fluent, accurate)*